# Transformation 00 — Target Binarization


In [ ]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from helpers.data_loader import DataLoader

train = pd.read_parquet(DataLoader.processed('train.parquet'))
val   = pd.read_parquet(DataLoader.processed('val.parquet'))
test  = pd.read_parquet(DataLoader.processed('test.parquet'))
print('Train shape:', train.shape)
print('Val   shape:', val.shape)
print('Test  shape:', test.shape)

## 1 · Class distribution BEFORE binarization

In [ ]:
print('=== Train — Results distribution (before) ===')
print(train['Results'].value_counts(normalize=True).mul(100).round(2))
print()
print('=== Val — Results distribution (before) ===')
print(val['Results'].value_counts(normalize=True).mul(100).round(2))
print()
print('=== Test — Results distribution (before) ===')
print(test['Results'].value_counts(normalize=True).mul(100).round(2))

## 2 · Binarize the target

In [ ]:
# -- Violation count (used to split PwC rows) --
def count_violations(series: pd.Series) -> pd.Series:
    return (
        series
        .fillna('')
        .apply(lambda x: len(x.split(' | ')) if x.strip() else 0)
    )

train['violation_count'] = count_violations(train['Violations'])
val['violation_count']   = count_violations(val['Violations'])
test['violation_count']  = count_violations(test['Violations'])

# Explore threshold on train PwC rows only (no leakage into val/test)
pwc_train = train[train['Results'] == 'Pass w/ Conditions']
print('PwC violation_count distribution:')
print(pwc_train['violation_count'].describe())
print()
for t in range(1, 8):
    frac_pass = (pwc_train['violation_count'] < t).mean()
    print(f'  threshold={t}:  {frac_pass:.1%} → Pass,  {1-frac_pass:.1%} → Fail')

In [ ]:
PwC_THRESHOLD = 4

TARGET_MAP = {'Pass': 0, 'Fail': 1}

def binarize_target(df):
    binary = df['Results'].map(TARGET_MAP)
    pwc_mask = df['Results'] == 'Pass w/ Conditions'
    binary[pwc_mask] = (df.loc[pwc_mask, 'violation_count'] >= PwC_THRESHOLD).astype(int)
    assert binary.isna().sum() == 0, 'Unmapped values!'
    return binary.astype(int)

train['Results'] = binarize_target(train)
val['Results']   = binarize_target(val)
test['Results']  = binarize_target(test)

print(f'Binarization complete (PwC threshold = {PwC_THRESHOLD}).')

## 3 · Class distribution AFTER binarization

In [ ]:
print('=== Train — Results distribution (after) ===')
print(train['Results'].value_counts(normalize=True).mul(100).round(2))
print(f'Class ratio (0:1): {(train["Results"]==0).sum() / (train["Results"]==1).sum():.2f}:1')
print()
print('=== Val — Results distribution (after) ===')
print(val['Results'].value_counts(normalize=True).mul(100).round(2))
print(f'Class ratio (0:1): {(val["Results"]==0).sum() / (val["Results"]==1).sum():.2f}:1')
print()
print('=== Test — Results distribution (after) ===')
print(test['Results'].value_counts(normalize=True).mul(100).round(2))
print(f'Class ratio (0:1): {(test["Results"]==0).sum() / (test["Results"]==1).sum():.2f}:1')

## 4 · Save intermediate results

In [ ]:
train.to_parquet(DataLoader.transformed('train_t00.parquet'), index=False)
val.to_parquet(DataLoader.transformed('val_t00.parquet'),   index=False)
test.to_parquet(DataLoader.transformed('test_t00.parquet'),  index=False)
print('Saved train_t00.parquet, val_t00.parquet, and test_t00.parquet to', DataLoader.TRANSFORMED_DIR)